# 🚀 Google Colab Setup, Sync & Secrets Provisioning Utility

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/konstantin-schekotihin/dl_course/blob/master/colab_setup.ipynb)

Welcome to the **Machine Learning & Deep Learning** course setup utility for Google Colab.

This utility automates:
1. **Mounting Google Drive** for persistent storage of your code, solutions, and trained model weights (`.pth`).
2. **Cloning & Synchronizing the Course Repository** (`dl_course`) with upstream GitHub.
3. **Secrets Provisioning** for **Weights & Biases (W&B)** and **Hugging Face** via Colab Secrets (🔑) or interactive login.
4. **Hardware Verification** (PyTorch, CUDA GPU acceleration).

## 1. Mount Google Drive & Clone / Sync Repository

Run the cell below to connect your Google Drive and sync the course repository:

In [ ]:
# @title 📦 1. Mount Drive & Sync Course Repository
import sys, os

if 'google.colab' in sys.modules:
    from google.colab import drive
    print("Step 1: Mounting Google Drive...")
    drive.mount('/content/drive')
    
    repo_url = "https://github.com/konstantin-schekotihin/dl_course.git"
    drive_dir = "/content/drive/MyDrive"
    repo_dir = os.path.join(drive_dir, "dl_course")
    
    if not os.path.exists(repo_dir):
        print(f"\nStep 2: Cloning repository into {repo_dir}...")
        !git clone {repo_url} "{repo_dir}"
        print("✅ Course repository cloned successfully into Google Drive!")
    else:
        print(f"\nStep 2: Repository found in Google Drive. Syncing latest updates from GitHub...")
        !git -C "{repo_dir}" pull --autostash
        print("✅ Course repository is up to date!")
        
    print("\nStep 3: Installing additional course dependencies...")
    !pip install -q torch-geometric wandb huggingface_hub
    print("✅ All dependencies installed!")
else:
    print("Running locally. No Colab setup required.")

## 2. Secrets & API Keys Provisioning (W&B, Hugging Face)

Several course modules (such as experiment tracking in `02_ML/01_classifiers-kNN.ipynb` and deep learning exercises in `03_DL/`) use **Weights & Biases (W&B)** and **Hugging Face**.

### Recommended: Add Secrets via Colab's 🔑 Secrets Tab
1. Click the **🔑 (Secrets)** icon in the left sidebar of Google Colab.
2. Add a new secret with:
   - **Name:** `WANDB_API_KEY` (Get your key from [wandb.ai/authorize](https://wandb.ai/authorize))
   - **Value:** *your_api_key*
   - Toggle **Notebook access** to **ON**.
3. *(Optional)* Add `HF_TOKEN` from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

Run the cell below to verify and configure your secrets:

In [ ]:
# @title 🔑 2. Configure & Verify Secrets (W&B / Hugging Face)
import sys, os

# 1. Weights & Biases (W&B)
try:
    import wandb
    wandb_key = None
    
    # Try retrieving from Google Colab userdata secrets
    if 'google.colab' in sys.modules:
        try:
            from google.colab import userdata
            wandb_key = userdata.get('WANDB_API_KEY')
        except Exception:
            pass
            
    if wandb_key:
        os.environ['WANDB_API_KEY'] = wandb_key
        wandb.login(key=wandb_key)
        print("✅ Weights & Biases: Authenticated successfully via Colab Secrets (WANDB_API_KEY).")
    else:
        print("ℹ️ WANDB_API_KEY not found in Colab Secrets.")
        print("   Prompting interactive login (click the link below to get your API key):")
        wandb.login()
        print("✅ Weights & Biases: Authenticated successfully interactively.")
except Exception as e:
    print(f"⚠️ W&B setup note: {e}")

print("-" * 50)

# 2. Hugging Face (Optional)
try:
    hf_token = None
    if 'google.colab' in sys.modules:
        try:
            from google.colab import userdata
            hf_token = userdata.get('HF_TOKEN')
        except Exception:
            pass
            
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        from huggingface_hub import login
        login(token=hf_token)
        print("✅ Hugging Face: Authenticated successfully via Colab Secrets (HF_TOKEN).")
    else:
        print("ℹ️ Hugging Face (HF_TOKEN): Not set in Colab Secrets (Optional, needed only for restricted HF models).")
except Exception as e:
    print(f"ℹ️ HF setup note: {e}")

## 3. Hardware & Environment Verification

Check GPU acceleration and verify that CUDA and PyTorch are functioning properly:

In [ ]:
# @title 🔍 3. Verify GPU Hardware & PyTorch
import torch

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"✅ GPU Acceleration Active: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"   CUDA Version: {torch.version.cuda}")
else:
    print("⚠️ Running on CPU.")
    print("   To enable GPU: Go to Runtime -> Change runtime type -> Select T4 GPU -> Save")

## 4. How to Open & Run Course Notebooks

Once synced to your Google Drive (`MyDrive/dl_course`), you can open any lecture notebook:

### Method A: Direct from Google Drive (Recommended)
1. Open [Google Drive](https://drive.google.com/).
2. Navigate to `MyDrive/dl_course/` $\rightarrow$ select the module folder (e.g. `02_ML/` or `03_DL/`).
3. Right-click any `.ipynb` file $\rightarrow$ **Open with $\rightarrow$ Google Colaboratory**.
4. All your code edits, cell outputs, and trained models will save automatically to your Drive.

### Method B: Direct from GitHub
1. In [Google Colab](https://colab.research.google.com/), go to **File $\rightarrow$ Open notebook $\rightarrow$ GitHub** tab.
2. Search `konstantin-schekotihin/dl_course`.
3. Click any lecture notebook to open it.

## 5. Maintenance & Reset Utility

If you ever encounter git conflicts or want to force a clean reset of the course materials to match upstream GitHub:

In [ ]:
# @title 🔄 4. Force Update / Reset Repository (Optional)
# Set reset_clean = True only if you want to discard local notebook changes and re-sync cleanly
reset_clean = False # @param {type:"boolean"}

repo_dir = "/content/drive/MyDrive/dl_course"
if os.path.exists(repo_dir):
    if reset_clean:
        print("Resetting local repository to clean upstream state...")
        !git -C "{repo_dir}" fetch origin
        !git -C "{repo_dir}" reset --hard origin/master
        print("✅ Reset complete.")
    else:
        !git -C "{repo_dir}" pull --autostash
        print("✅ Repository updated.")
else:
    print("Repository not found. Run Step 1 first.")